In [6]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------
gauss = pd.read_csv("/yunity/tkerby2/Local_CorEx/Local_CorEx_Experiments/ablation_data_v2/gaussian_ablation.csv")
nonlin = pd.read_csv("/yunity/tkerby2/Local_CorEx/Local_CorEx_Experiments/ablation_data_v2/nonlinear_ablation.csv")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

In [ ]:
# Optional: make the labels a little easier to read
for df in [gauss, nonlin]:
    if "model_kind" in df.columns:
        df["model_kind"] = df["model_kind"].replace({"linear": "Linear CorEx", "bio": "Bio CorEx"})
    if "partition_method" in df.columns:
        df["partition_method"] = df["partition_method"].replace(
            {
                "global": "Global",
                "oracle": "Oracle",
                "phate": "PHATE+kmeans",
                "raw_kmeans": "Raw+kmeans",
                "random": "Random",
            }
        )
        
# ---------------------------------------------------------------------
# Utility
# ---------------------------------------------------------------------
METRICS = [
    "ave_group_cosine",
    "ave_auc_group_prc",
    "ave_lf_cosine",
    "ave_auc_lf_prc",
]

def show_groupby(df, group_cols, metric_cols=METRICS, agg="mean", sort_by=None):
    out = (
        df.groupby(group_cols, dropna=False)[metric_cols]
        .agg(agg)
        .reset_index()
    )
    if sort_by is not None:
        out = out.sort_values(sort_by, ascending=False)
    return out

def print_section(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

def best_rows(df, sort_metric="ave_auc_lf_prc", n=10, cols=None):
    if cols is None:
        cols = ["dataset", "partition_method", "model_kind", "cluster_size", "alpha", "n_noise", sort_metric]
    return df.sort_values(sort_metric, ascending=False)[cols].head(n)

In [10]:
# ---------------------------------------------------------------------
# 1) Main story: global vs local
# ---------------------------------------------------------------------
print_section("GAUSSIAN: Global vs Local summary")
gauss_global_local = show_groupby(
    gauss,
    ["partition_method", "model_kind", "disjoint"],
    sort_by="ave_auc_lf_prc",
)
print(gauss_global_local)

print_section("NONLINEAR: Global vs Local summary")
nonlin_global_local = show_groupby(
    nonlin,
    ["partition_method", "model_kind"],
    sort_by="ave_auc_lf_prc",
)
print(nonlin_global_local)


GAUSSIAN: Global vs Local summary
   partition_method    model_kind  disjoint  ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc
2            Global  Linear CorEx     False          0.123171           0.976414       0.087664        0.989282
7            Oracle  Linear CorEx      True          0.147018           0.970766       0.159003        0.966898
5            Oracle     Bio CorEx      True          0.265547           0.813203       0.144635        0.934716
0            Global     Bio CorEx     False          0.226762           0.859989       0.142609        0.934393
14           Random  Linear CorEx     False          0.182285           0.922823       0.174980        0.931258
6            Oracle  Linear CorEx     False          0.188265           0.925363       0.194556        0.921150
10     PHATE+kmeans  Linear CorEx     False          0.207324           0.909255       0.209356        0.908745
4            Oracle     Bio CorEx     False          0.218208        

In [11]:
# ---------------------------------------------------------------------
# 2) Partition quality story
# ---------------------------------------------------------------------
print_section("GAUSSIAN: Partition quality by method, sample size, and alpha")
gauss_partition = show_groupby(
    gauss[gauss["partition_method"].isin(["Oracle", "PHATE+kmeans", "Raw+kmeans", "Random"])],
    ["partition_method", "cluster_size", "alpha", "disjoint"],
    sort_by="ave_auc_lf_prc",
)
print(gauss_partition)

print_section("NONLINEAR: Partition quality by method and sample size")
nonlin_partition = show_groupby(
    nonlin[nonlin["partition_method"].isin(["Oracle", "PHATE+kmeans", "Raw+kmeans", "Random"])],
    ["partition_method", "cluster_size"],
    sort_by="ave_auc_lf_prc",
)
print(nonlin_partition)


GAUSSIAN: Partition quality by method, sample size, and alpha
   partition_method  cluster_size  alpha  disjoint  ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc
1            Oracle            50    0.0      True          0.182952           0.928219       0.158047        0.956952
15       Raw+kmeans            50    1.0      True          0.154753           0.929624       0.134812        0.954763
3            Oracle            50    1.0      True          0.229613           0.855750       0.145591        0.944662
7      PHATE+kmeans            50    1.0      True          0.197926           0.883544       0.158967        0.932523
6      PHATE+kmeans            50    1.0     False          0.220395           0.857818       0.158841        0.925840
0            Oracle            50    0.0     False          0.210711           0.883843       0.181604        0.920804
8            Random            50    0.0     False          0.239587           0.847625       0.178849  

In [12]:
# ---------------------------------------------------------------------
# 3) Sample size story
# ---------------------------------------------------------------------
print_section("GAUSSIAN: Sample size effects")
gauss_size = show_groupby(
    gauss,
    ["cluster_size", "partition_method", "model_kind"],
    sort_by="ave_auc_lf_prc",
)
print(gauss_size)

print_section("NONLINEAR: Sample size effects")
nonlin_size = show_groupby(
    nonlin,
    ["cluster_size", "partition_method", "model_kind"],
    sort_by="ave_auc_lf_prc",
)
print(nonlin_size)


GAUSSIAN: Sample size effects
   cluster_size partition_method    model_kind  ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc
3            50           Oracle  Linear CorEx          0.167642           0.948065       0.176779        0.944024
2            50           Oracle     Bio CorEx          0.241878           0.826928       0.153987        0.921359
1            50           Global  Linear CorEx          0.294401           0.784223       0.199784        0.904395
9            50       Raw+kmeans  Linear CorEx          0.258002           0.844499       0.251038        0.857811
5            50     PHATE+kmeans  Linear CorEx          0.270572           0.829171       0.261469        0.850978
0            50           Global     Bio CorEx          0.352934           0.690200       0.231924        0.844848
4            50     PHATE+kmeans     Bio CorEx          0.333234           0.716371       0.256325        0.800957
8            50       Raw+kmeans     Bio CorEx   

In [13]:
# ---------------------------------------------------------------------
# 4) Alpha story for Gaussian
# ---------------------------------------------------------------------
print_section("GAUSSIAN: Alpha effects")
gauss_alpha = show_groupby(
    gauss,
    ["alpha", "partition_method", "model_kind", "disjoint"],
    sort_by="ave_auc_lf_prc",
)
print(gauss_alpha)

# If you want just the best partition method per alpha:
print_section("GAUSSIAN: Best method within each alpha")
gauss_best_by_alpha = (
    gauss.groupby(["alpha", "disjoint", "partition_method", "model_kind"], dropna=False)[METRICS]
    .mean()
    .reset_index()
    .sort_values(["alpha", "ave_auc_lf_prc"], ascending=[True, False])
)
print(gauss_best_by_alpha)


GAUSSIAN: Alpha effects
    alpha partition_method    model_kind  disjoint  ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc
2     0.0           Global  Linear CorEx     False          0.134041           0.994886       0.083580        1.000000
22    1.0           Global  Linear CorEx     False          0.112302           0.957942       0.091748        0.978563
7     0.0           Oracle  Linear CorEx      True          0.159963           0.979143       0.173427        0.977310
39    1.0       Raw+kmeans  Linear CorEx      True          0.134074           0.962389       0.144579        0.956486
27    1.0           Oracle  Linear CorEx      True          0.134074           0.962389       0.144579        0.956486
31    1.0     PHATE+kmeans  Linear CorEx      True          0.134074           0.962389       0.144579        0.956486
37    1.0       Raw+kmeans     Bio CorEx      True          0.175432           0.896859       0.125045        0.953040
0     0.0           Glo

In [14]:
# ---------------------------------------------------------------------
# 5) Nuisance variables story
# ---------------------------------------------------------------------
print_section("GAUSSIAN: Nuisance-variable effect")
gauss_noise = show_groupby(
    gauss,
    ["n_noise", "cluster_size", "partition_method", "model_kind"],
    sort_by="ave_auc_lf_prc",
)
print(gauss_noise)

print_section("NONLINEAR: Nuisance-variable effect")
nonlin_noise = show_groupby(
    nonlin,
    ["n_noise", "cluster_size", "partition_method", "model_kind"],
    sort_by="ave_auc_lf_prc",
)
print(nonlin_noise)


GAUSSIAN: Nuisance-variable effect
   n_noise  cluster_size partition_method    model_kind  ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc
3        0            50           Oracle  Linear CorEx          0.167642           0.948065       0.176779        0.944024
2        0            50           Oracle     Bio CorEx          0.241878           0.826928       0.153987        0.921359
1        0            50           Global  Linear CorEx          0.294401           0.784223       0.199784        0.904395
9        0            50       Raw+kmeans  Linear CorEx          0.258002           0.844499       0.251038        0.857811
5        0            50     PHATE+kmeans  Linear CorEx          0.270572           0.829171       0.261469        0.850978
0        0            50           Global     Bio CorEx          0.352934           0.690200       0.231924        0.844848
4        0            50     PHATE+kmeans     Bio CorEx          0.333234           0.716371    

In [15]:
# ---------------------------------------------------------------------
# 6) Oracle vs PHATE vs Random directly
# ---------------------------------------------------------------------
print_section("GAUSSIAN: Oracle vs PHATE vs Random vs Raw kmeans")
gauss_compare = (
    gauss[gauss["partition_method"].isin(["Oracle", "PHATE+kmeans", "Raw+kmeans", "Random"])]
    .groupby(["partition_method", "cluster_size", "alpha", "disjoint", "model_kind"])[METRICS]
    .mean()
    .reset_index()
    .sort_values(["partition_method", "cluster_size", "alpha", "ave_auc_lf_prc"], ascending=[True, True, True, False])
)
print(gauss_compare)

print_section("NONLINEAR: Oracle vs PHATE vs Random vs Raw kmeans")
nonlin_compare = (
    nonlin[nonlin["partition_method"].isin(["Oracle", "PHATE+kmeans", "Raw+kmeans", "Random"])]
    .groupby(["partition_method", "cluster_size", "model_kind"])[METRICS]
    .mean()
    .reset_index()
    .sort_values(["partition_method", "cluster_size", "ave_auc_lf_prc"], ascending=[True, True, False])
)
print(nonlin_compare)


GAUSSIAN: Oracle vs PHATE vs Random vs Raw kmeans
   partition_method  cluster_size  alpha  disjoint    model_kind  ave_group_cosine  ave_auc_group_prc  ave_lf_cosine  ave_auc_lf_prc
3            Oracle            50    0.0      True  Linear CorEx          0.159963           0.979143       0.173427        0.977310
2            Oracle            50    0.0      True     Bio CorEx          0.205941           0.877295       0.142666        0.936595
1            Oracle            50    0.0     False  Linear CorEx          0.169170           0.948228       0.187006        0.936202
0            Oracle            50    0.0     False     Bio CorEx          0.252251           0.819458       0.176201        0.905407
7            Oracle            50    1.0      True  Linear CorEx          0.134074           0.962389       0.144579        0.956486
6            Oracle            50    1.0      True     Bio CorEx          0.325153           0.749110       0.146603        0.932837
4            Oracl

In [17]:
# ---------------------------------------------------------------------
# 7) Quick views of the strongest rows
# ---------------------------------------------------------------------
print_section("Top Gaussian rows by ave_auc_lf_prc")
print(best_rows(gauss, sort_metric="ave_auc_lf_prc", n=15))

print_section("Top Nonlinear rows by ave_auc_lf_prc")
# print(best_rows(nonlin, sort_metric="ave_auc_lf_prc", n=15))


Top Gaussian rows by ave_auc_lf_prc
     dataset partition_method    model_kind  cluster_size  alpha  n_noise  ave_auc_lf_prc
0   gaussian           Global  Linear CorEx            50    0.0        0        1.000000
10  gaussian           Global  Linear CorEx            50    1.0        0        0.978563
22  gaussian           Oracle  Linear CorEx            50    0.0        0        0.977310
36  gaussian       Raw+kmeans  Linear CorEx            50    1.0        0        0.956486
34  gaussian     PHATE+kmeans  Linear CorEx            50    1.0        0        0.956486
32  gaussian           Oracle  Linear CorEx            50    1.0        0        0.956486
37  gaussian       Raw+kmeans     Bio CorEx            50    1.0        0        0.953040
1   gaussian           Global     Bio CorEx            50    0.0        0        0.952972
15  gaussian     PHATE+kmeans     Bio CorEx            50    1.0        0        0.945581
8   gaussian           Random  Linear CorEx            50    0.

In [18]:
# ---------------------------------------------------------------------
# 8) Optional compact tables for paper drafting
# ---------------------------------------------------------------------
print_section("Compact Gaussian table: partition method x sample size")
gauss_compact = (
    gauss.groupby(["partition_method", "cluster_size"])[["ave_auc_lf_prc", "ave_lf_cosine", "partition_ari", "cluster_purity"]]
    .mean()
    .reset_index()
    .sort_values(["partition_method", "cluster_size"])
)
print(gauss_compact)

print_section("Compact Nonlinear table: partition method x sample size")
nonlin_compact = (
    nonlin.groupby(["partition_method", "cluster_size"])[["ave_auc_lf_prc", "ave_lf_cosine", "partition_ari", "cluster_purity"]]
    .mean()
    .reset_index()
    .sort_values(["partition_method", "cluster_size"])
)
print(nonlin_compact)


Compact Gaussian table: partition method x sample size
  partition_method  cluster_size  ave_auc_lf_prc  ave_lf_cosine  partition_ari  cluster_purity
0           Global            50        0.874621       0.215854            NaN             NaN
1           Oracle            50        0.932692       0.165383       1.000000          1.0000
2     PHATE+kmeans            50        0.825968       0.258897       0.503849          0.7700
3           Random            50        0.775892       0.294514       0.000000          0.5500
4       Raw+kmeans            50        0.826654       0.252125       0.502392          0.7775

Compact Nonlinear table: partition method x sample size
  partition_method  cluster_size  ave_auc_lf_prc  ave_lf_cosine  partition_ari  cluster_purity
0           Global            50        0.468518       0.484627            NaN             NaN
1           Oracle            50        0.442030       0.615805       1.000000            1.00
2     PHATE+kmeans            50